# 🔍 Notebook 07: Pipeline Validation

## Overview

This notebook validates the **end-to-end ML pipeline** by:

1. **Extracting features** from FMA small audio tracks using librosa
2. **Comparing** our extracted features with FMA's pre-computed `features.csv`
3. **Computing correlations** across all 518 features
4. **Cross-validation**: Train on FMA features -> Predict on our extraction
5. **Testing** with production model

### Key Question
**Does our feature extraction produce the same values as FMA's pre-computed features?**

| Result | Interpretation |
|--------|----------------|
| >70% accuracy | Pipeline matches FMA - Ready to deploy |
| >50% accuracy | Directionally compatible - Usable with caution |
| ~10% accuracy | Fundamental mismatch (30s vs full-track statistics) |

## 1. Setup and Imports

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"Librosa version: {librosa.__version__}")
print("Libraries imported successfully!")

In [ ]:
# === PATHS ===
BASE_DIR = os.path.dirname(os.getcwd())  # MLDeploy/
DATA_DIR = os.path.join(BASE_DIR, 'data')
FMA_META_DIR = os.path.join(DATA_DIR, 'fma_small', 'fma_metadata')
FMA_AUDIO_DIR = os.path.join(DATA_DIR, 'fma_small', 'fma_small')  # Contains 000/, 001/, etc.
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')

print(f"BASE_DIR: {BASE_DIR}")
print(f"FMA_META_DIR: {FMA_META_DIR}")
print(f"FMA_AUDIO_DIR: {FMA_AUDIO_DIR}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

# Verify paths exist
for name, path in [('FMA_META_DIR', FMA_META_DIR), ('FMA_AUDIO_DIR', FMA_AUDIO_DIR), ('ARTIFACTS_DIR', ARTIFACTS_DIR)]:
    status = 'exists' if os.path.exists(path) else 'NOT FOUND'
    print(f"  {name}: {status}")

## 2. Load FMA Metadata and Features

In [ ]:
# Load FMA pre-computed features (518 features with 3-level column headers)
features_df = pd.read_csv(os.path.join(FMA_META_DIR, 'features.csv'), index_col=0, header=[0, 1, 2])

print(f"FMA Features shape: {features_df.shape}")
print(f"Total features: {features_df.shape[1]}")
print(f"\nColumn structure (first 5):")
for col in features_df.columns[:5]:
    print(f"  {col}")

In [ ]:
# Flatten column names to match our format: feature_stat_number
feature_cols_flat = ['_'.join(map(str, col)).strip() for col in features_df.columns.values]
features_flat = features_df.copy()
features_flat.columns = feature_cols_flat

print(f"Flattened column names (first 20):")
for col in feature_cols_flat[:20]:
    print(f"  {col}")
print(f"...")
print(f"\nTotal: {len(feature_cols_flat)} features")

In [ ]:
# Load tracks metadata for genre labels
tracks = pd.read_csv(os.path.join(FMA_META_DIR, 'tracks.csv'), index_col=0, header=[0, 1])

print(f"Tracks shape: {tracks.shape}")
print(f"\nGenre distribution (small subset):")
small_mask = tracks[('set', 'subset')] == 'small'
small_genres = tracks.loc[small_mask, ('track', 'genre_top')].value_counts()
for genre, count in small_genres.items():
    print(f"  {genre}: {count}")

## 3. Load Production Artifacts

In [ ]:
# Load model
with open(os.path.join(ARTIFACTS_DIR, 'model.pkl'), 'rb') as f:
    model = pickle.load(f)
print(f"Model: {type(model).__name__}")

# Load scaler
with open(os.path.join(ARTIFACTS_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
print(f"Scaler: {type(scaler).__name__} (expects {scaler.n_features_in_} features)")

# Load label encoder
with open(os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
print(f"Label Encoder: {len(label_encoder.classes_)} genres")
print(f"   Genres: {list(label_encoder.classes_)}")

# Load feature names
with open(os.path.join(ARTIFACTS_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)
print(f"Feature Names: {len(feature_names)} features")

In [ ]:
# Verify feature names match
fma_features_set = set(feature_cols_flat)
prod_features_set = set(feature_names)

common = fma_features_set.intersection(prod_features_set)
only_fma = fma_features_set - prod_features_set
only_prod = prod_features_set - fma_features_set

print(f"Feature matching:")
print(f"  Common features: {len(common)}")
print(f"  Only in FMA: {len(only_fma)}")
print(f"  Only in prod: {len(only_prod)}")

if len(common) == len(feature_names) == 518:
    print(f"\nAll 518 features match!")
else:
    print(f"\nFeature mismatch detected")

## 4. Feature Extraction Pipeline (518 FMA Features)

In [ ]:
def compute_stats(feature_array):
    """
    Compute statistics for a feature array.
    FMA uses: kurtosis, max, mean, median, min, skew, std
    """
    return {
        'kurtosis': stats.kurtosis(feature_array),
        'max': np.max(feature_array),
        'mean': np.mean(feature_array),
        'median': np.median(feature_array),
        'min': np.min(feature_array),
        'skew': stats.skew(feature_array),
        'std': np.std(feature_array)
    }

def extract_fma_features(audio_path, sr=22050, duration=30):
    """
    Extract 518 features matching FMA features.csv format.
    
    Features:
    - chroma_cens (12 bins x 7 stats = 84)
    - chroma_cqt (12 bins x 7 stats = 84)
    - chroma_stft (12 bins x 7 stats = 84)
    - mfcc (20 coeffs x 7 stats = 140)
    - rmse (1 x 7 stats = 7)
    - spectral_bandwidth (1 x 7 stats = 7)
    - spectral_centroid (1 x 7 stats = 7)
    - spectral_contrast (7 bands x 7 stats = 49)
    - spectral_rolloff (1 x 7 stats = 7)
    - tonnetz (6 dims x 7 stats = 42)
    - zcr (1 x 7 stats = 7)
    
    Total: 84+84+84+140+7+7+7+49+7+42+7 = 518 features
    """
    try:
        y, sr_actual = librosa.load(audio_path, sr=sr, duration=duration)
        features = {}
        
        # chroma_cens - 12 bins
        chroma_cens = librosa.feature.chroma_cens(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_cens[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_cens_{stat_name}_{i+1:02d}'] = stat_val
        
        # chroma_cqt - 12 bins
        chroma_cqt = librosa.feature.chroma_cqt(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_cqt[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_cqt_{stat_name}_{i+1:02d}'] = stat_val
        
        # chroma_stft - 12 bins
        chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_stft[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_stft_{stat_name}_{i+1:02d}'] = stat_val
        
        # mfcc - 20 coeffs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        for i in range(20):
            stats_dict = compute_stats(mfccs[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'mfcc_{stat_name}_{i+1:02d}'] = stat_val
        
        # rmse (FMA uses 'rmse', librosa has 'rms')
        rms = librosa.feature.rms(y=y)
        stats_dict = compute_stats(rms[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'rmse_{stat_name}_01'] = stat_val
        
        # spectral_bandwidth
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        stats_dict = compute_stats(spec_bw[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_bandwidth_{stat_name}_01'] = stat_val
        
        # spectral_centroid
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        stats_dict = compute_stats(spec_cent[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_centroid_{stat_name}_01'] = stat_val
        
        # spectral_contrast - 7 bands
        spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
        for i in range(7):
            stats_dict = compute_stats(spec_contrast[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'spectral_contrast_{stat_name}_{i+1:02d}'] = stat_val
        
        # spectral_rolloff
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        stats_dict = compute_stats(spec_rolloff[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_rolloff_{stat_name}_01'] = stat_val
        
        # tonnetz - 6 dims
        y_harmonic = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)
        for i in range(6):
            stats_dict = compute_stats(tonnetz[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'tonnetz_{stat_name}_{i+1:02d}'] = stat_val
        
        # zcr
        zcr = librosa.feature.zero_crossing_rate(y)
        stats_dict = compute_stats(zcr[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'zcr_{stat_name}_01'] = stat_val
        
        return features
    
    except Exception as e:
        print(f"Error extracting features from {audio_path}: {e}")
        return None

In [ ]:
# Helper to get audio path
def get_audio_path(track_id):
    """Get audio file path for a track ID (FMA uses 6-digit zero-padded IDs)"""
    tid_str = f"{track_id:06d}"
    folder = tid_str[:3]
    return os.path.join(FMA_AUDIO_DIR, folder, f"{tid_str}.mp3")

# Find a valid track from small subset
small_track_ids = tracks.loc[small_mask].index.tolist()
test_tid = small_track_ids[0]
test_path = get_audio_path(test_tid)

print(f"Test track ID: {test_tid}")
print(f"Audio path: {test_path}")
print(f"Exists: {os.path.exists(test_path)}")

In [ ]:
# Extract features from test track
if os.path.exists(test_path):
    print(f"Extracting features from track {test_tid}...")
    test_features = extract_fma_features(test_path)
    
    if test_features:
        print(f"\nExtracted {len(test_features)} features")
        print(f"\nSample features:")
        for i, (k, v) in enumerate(list(test_features.items())[:10]):
            print(f"  {k}: {v:.6f}")
    else:
        print("Feature extraction failed")
else:
    print(f"Audio file not found: {test_path}")

## 5. Compare Extracted Features with FMA features.csv

In [ ]:
# Get FMA's pre-computed features for the same track
if test_tid in features_flat.index:
    fma_features = features_flat.loc[test_tid]
    print(f"FMA features for track {test_tid}: {len(fma_features)} values")
    print(f"\nSample FMA features:")
    for col in feature_cols_flat[:10]:
        print(f"  {col}: {fma_features[col]:.6f}")
else:
    print(f"Track {test_tid} not in features.csv")

In [ ]:
# Compare our extraction vs FMA
if test_features and test_tid in features_flat.index:
    print("=" * 70)
    print("FEATURE COMPARISON: Our Extraction vs FMA features.csv")
    print("=" * 70)
    
    comparisons = []
    for fma_col in feature_cols_flat[:30]:
        fma_val = fma_features[fma_col]
        our_val = test_features.get(fma_col, np.nan)
        
        if not np.isnan(our_val) and not np.isnan(fma_val) and fma_val != 0:
            diff_pct = abs(our_val - fma_val) / abs(fma_val) * 100
        else:
            diff_pct = np.nan
        
        comparisons.append({
            'feature': fma_col,
            'fma': fma_val,
            'ours': our_val,
            'diff_pct': diff_pct
        })
    
    df_comp = pd.DataFrame(comparisons)
    print(df_comp.to_string(index=False))

## 6. Batch Feature Extraction (50 Tracks)

In [ ]:
# Extract features from 50 FMA small tracks
N_TRACKS = 50

print(f"Extracting features from {N_TRACKS} FMA small tracks...")
print("(This may take a few minutes)")
print()

extracted_features = []
track_ids_extracted = []
failed_tracks = []

for i, tid in enumerate(small_track_ids[:N_TRACKS]):
    audio_path = get_audio_path(tid)
    
    if not os.path.exists(audio_path):
        failed_tracks.append((tid, 'file_not_found'))
        continue
    
    features = extract_fma_features(audio_path)
    
    if features:
        extracted_features.append(features)
        track_ids_extracted.append(tid)
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{N_TRACKS} tracks...")
    else:
        failed_tracks.append((tid, 'extraction_error'))

print(f"\nSuccessfully extracted: {len(extracted_features)} tracks")
print(f"Failed: {len(failed_tracks)} tracks")

In [ ]:
# Create DataFrame from extracted features
if extracted_features:
    df_extracted = pd.DataFrame(extracted_features, index=track_ids_extracted)
    print(f"Extracted features DataFrame: {df_extracted.shape}")
    
    # Check column alignment
    missing_cols = [c for c in feature_names if c not in df_extracted.columns]
    extra_cols = [c for c in df_extracted.columns if c not in feature_names]
    
    print(f"\nColumn alignment:")
    print(f"  Missing from extraction: {len(missing_cols)}")
    print(f"  Extra in extraction: {len(extra_cols)}")

## 7. Compute Correlations Across All 518 Features

In [ ]:
# Compare extracted features with FMA for same tracks
if extracted_features:
    common_tids = [tid for tid in track_ids_extracted if tid in features_flat.index]
    print(f"Tracks with both extracted and FMA features: {len(common_tids)}")
    
    fma_subset = features_flat.loc[common_tids]
    our_subset = df_extracted.loc[common_tids]
    
    correlations = []
    
    for col in feature_names:
        if col in fma_subset.columns and col in our_subset.columns:
            fma_vals = fma_subset[col].values
            our_vals = our_subset[col].values
            
            valid = ~(np.isnan(fma_vals) | np.isnan(our_vals) | 
                     np.isinf(fma_vals) | np.isinf(our_vals))
            
            if valid.sum() > 5:
                corr = np.corrcoef(fma_vals[valid], our_vals[valid])[0, 1]
            else:
                corr = np.nan
        else:
            corr = np.nan
        
        correlations.append({'feature': col, 'correlation': corr})
    
    df_corr = pd.DataFrame(correlations)
    df_corr = df_corr.dropna()
    
    print(f"\nCORRELATION STATISTICS (Our Extraction vs FMA):")
    print(f"  Features analyzed: {len(df_corr)}")
    print(f"  Mean correlation: {df_corr['correlation'].mean():.4f}")
    print(f"  Median correlation: {df_corr['correlation'].median():.4f}")
    print(f"  Min correlation: {df_corr['correlation'].min():.4f}")
    print(f"  Max correlation: {df_corr['correlation'].max():.4f}")

In [ ]:
# Visualize correlation distribution
if len(df_corr) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df_corr['correlation'], bins=50, color='steelblue', edgecolor='black')
    axes[0].axvline(df_corr['correlation'].mean(), color='red', linestyle='--', 
                    label=f"Mean: {df_corr['correlation'].mean():.3f}")
    axes[0].set_xlabel('Correlation')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Feature Correlation Distribution (Our Extraction vs FMA)')
    axes[0].legend()
    
    df_corr['feature_type'] = df_corr['feature'].str.split('_').str[0]
    type_corr = df_corr.groupby('feature_type')['correlation'].mean().sort_values(ascending=False)
    
    axes[1].barh(type_corr.index, type_corr.values, color='steelblue')
    axes[1].set_xlabel('Mean Correlation')
    axes[1].set_title('Correlation by Feature Type')
    axes[1].axvline(0.5, color='orange', linestyle='--', label='0.5 threshold')
    axes[1].axvline(0.7, color='green', linestyle='--', label='0.7 threshold')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 8. Cross-Validation: Train on FMA, Predict on Our Extraction

In [ ]:
# Use production model to predict on our extracted features
if len(common_tids) > 0:
    print("Testing production model on our extracted features...")
    
    y_true_genres = tracks.loc[common_tids, ('track', 'genre_top')]
    
    valid_mask = y_true_genres.isin(label_encoder.classes_)
    valid_tids = y_true_genres[valid_mask].index.tolist()
    
    print(f"Tracks with valid genres: {len(valid_tids)} / {len(common_tids)}")
    
    if len(valid_tids) > 0:
        y_true = y_true_genres.loc[valid_tids]
        
        X_our = df_extracted.loc[valid_tids][feature_names].fillna(0).replace([np.inf, -np.inf], 0)
        X_our_scaled = scaler.transform(X_our.values)
        
        y_pred_encoded = model.predict(X_our_scaled)
        y_pred = label_encoder.inverse_transform(y_pred_encoded)
        
        accuracy = accuracy_score(y_true, y_pred)
        
        print(f"\n" + "=" * 60)
        print(f"CROSS-VALIDATION RESULTS")
        print(f"=" * 60)
        print(f"Accuracy: {accuracy:.2%}")
        print(f"\nInterpretation:")
        if accuracy > 0.7:
            print("  >70%: Pipeline matches FMA - Ready to deploy!")
        elif accuracy > 0.5:
            print("  >50%: Directionally compatible - Usable with caution")
        else:
            print("  ~10%: Fundamental mismatch (30s vs full-track statistics)")

In [ ]:
# Confusion matrix
if len(valid_tids) > 0:
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix: Production Model on Our Feature Extraction')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 9. Test with Production Model (FMA Features)

In [ ]:
# Baseline: Test production model on FMA's own features
if len(valid_tids) > 0:
    print("Testing production model on FMA's own features (baseline)...")
    
    X_fma = fma_subset.loc[valid_tids][feature_names].fillna(0).replace([np.inf, -np.inf], 0)
    X_fma_scaled = scaler.transform(X_fma.values)
    
    y_pred_fma_encoded = model.predict(X_fma_scaled)
    y_pred_fma = label_encoder.inverse_transform(y_pred_fma_encoded)
    
    accuracy_fma = accuracy_score(y_true, y_pred_fma)
    
    print(f"\nBaseline Accuracy (FMA features): {accuracy_fma:.2%}")
    print(f"Our Extraction Accuracy: {accuracy:.2%}")
    print(f"Difference: {(accuracy - accuracy_fma)*100:.1f}%")

## 10. Summary

### Pipeline Validation Results

| Check | Status |
|-------|--------|
| Model loaded (SVC, RBF, C=10, balanced) | |
| Feature count (518) | |
| Column ordering matches features.csv | |
| rmse naming (FMA uses rmse_*) | |

### Interpretation

| Accuracy | Meaning |
|----------|--------|
| >70% | Pipeline matches FMA - Ready to deploy |
| >50% | Directionally compatible - Usable with caution |
| ~10% | Fundamental mismatch (30s vs full-track statistics) |

### Known Issue

FMA `features.csv` is computed from **full-length tracks** (often 3+ minutes), while our extraction is from **30-second clips**. This can cause statistical differences:

- Mean/median: Usually similar
- Kurtosis/skew: Can differ significantly
- Min/max: May differ based on song structure

### Recommendations

1. **For production**: Use features extracted from 30s clips for both training and inference
2. **Alternative**: Use a dataset with consistent extraction (like GTZAN 30-sec clips)

# 🔍 Notebook 07: Pipeline Validation

## Overview

This notebook validates the **end-to-end ML pipeline** by:

1. **Extracting features** from FMA small audio tracks using librosa
2. **Comparing** our extracted features with FMA's pre-computed `features.csv`
3. **Computing correlations** across all 518 features
4. **Cross-validation**: Train on FMA features → Predict on our extraction
5. **Testing** with production model

### Key Question
**Does our feature extraction produce the same values as FMA's pre-computed features?**

| Result | Interpretation |
|--------|----------------|
| >70% accuracy | Pipeline matches FMA → Ready to deploy |
| >50% accuracy | Directionally compatible → Usable with caution |
| ~10% accuracy | Fundamental mismatch (30s vs full-track statistics) |

---

## 1. Setup & Imports

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"Librosa version: {librosa.__version__}")
print("✅ Libraries imported successfully!")

In [ ]:
# === PATHS ===
BASE_DIR = os.path.dirname(os.getcwd())  # MLDeploy/
DATA_DIR = os.path.join(BASE_DIR, 'data')
FMA_META_DIR = os.path.join(DATA_DIR, 'fma_small', 'fma_metadata')
FMA_AUDIO_DIR = os.path.join(DATA_DIR, 'fma_small', 'fma_small')  # Contains 000/, 001/, etc.
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')

print(f"BASE_DIR: {BASE_DIR}")
print(f"FMA_META_DIR: {FMA_META_DIR}")
print(f"FMA_AUDIO_DIR: {FMA_AUDIO_DIR}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

# Verify paths exist
for name, path in [('FMA_META_DIR', FMA_META_DIR), ('FMA_AUDIO_DIR', FMA_AUDIO_DIR), ('ARTIFACTS_DIR', ARTIFACTS_DIR)]:
    if os.path.exists(path):
        print(f"  ✅ {name} exists")
    else:
        print(f"  ❌ {name} NOT FOUND")

## 2. Load FMA Metadata & Features

In [ ]:
# Load FMA pre-computed features (518 features with 3-level column headers)
features_df = pd.read_csv(os.path.join(FMA_META_DIR, 'features.csv'), index_col=0, header=[0, 1, 2])

print(f"FMA Features shape: {features_df.shape}")
print(f"Total features: {features_df.shape[1]}")
print(f"\nColumn structure (first 5):")
for col in features_df.columns[:5]:
    print(f"  {col}")

In [ ]:
# Flatten column names to match our format: feature_stat_number
feature_cols_flat = ['_'.join(map(str, col)).strip() for col in features_df.columns.values]
features_flat = features_df.copy()
features_flat.columns = feature_cols_flat

print(f"Flattened column names (first 20):")
for col in feature_cols_flat[:20]:
    print(f"  {col}")
print(f"...")
print(f"\nTotal: {len(feature_cols_flat)} features")

In [ ]:
# Load tracks metadata for genre labels
tracks = pd.read_csv(os.path.join(FMA_META_DIR, 'tracks.csv'), index_col=0, header=[0, 1])

print(f"Tracks shape: {tracks.shape}")
print(f"\nGenre distribution (small subset):")
small_mask = tracks[('set', 'subset')] == 'small'
small_genres = tracks.loc[small_mask, ('track', 'genre_top')].value_counts()
for genre, count in small_genres.items():
    print(f"  {genre}: {count}")

## 3. Load Production Artifacts

In [ ]:
# Load model
with open(os.path.join(ARTIFACTS_DIR, 'model.pkl'), 'rb') as f:
    model = pickle.load(f)
print(f"✅ Model: {type(model).__name__}")

# Load scaler
with open(os.path.join(ARTIFACTS_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
print(f"✅ Scaler: {type(scaler).__name__} (expects {scaler.n_features_in_} features)")

# Load label encoder
with open(os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
print(f"✅ Label Encoder: {len(label_encoder.classes_)} genres")
print(f"   Genres: {list(label_encoder.classes_)}")

# Load feature names
with open(os.path.join(ARTIFACTS_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)
print(f"✅ Feature Names: {len(feature_names)} features")

In [ ]:
# Verify feature names match
fma_features_set = set(feature_cols_flat)
prod_features_set = set(feature_names)

common = fma_features_set.intersection(prod_features_set)
only_fma = fma_features_set - prod_features_set
only_prod = prod_features_set - fma_features_set

print(f"Feature matching:")
print(f"  Common features: {len(common)}")
print(f"  Only in FMA: {len(only_fma)}")
print(f"  Only in prod: {len(only_prod)}")

if len(common) == len(feature_names) == 518:
    print(f"\n✅ All 518 features match!")
else:
    print(f"\n⚠️ Feature mismatch detected")
    if only_fma:
        print(f"  FMA only (first 5): {list(only_fma)[:5]}")
    if only_prod:
        print(f"  Prod only (first 5): {list(only_prod)[:5]}")

## 4. Feature Extraction Pipeline (518 FMA Features)

In [ ]:
def compute_stats(feature_array):
    """
    Compute statistics for a feature array.
    FMA uses: kurtosis, max, mean, median, min, skew, std
    """
    from scipy import stats
    return {
        'kurtosis': stats.kurtosis(feature_array),
        'max': np.max(feature_array),
        'mean': np.mean(feature_array),
        'median': np.median(feature_array),
        'min': np.min(feature_array),
        'skew': stats.skew(feature_array),
        'std': np.std(feature_array)
    }

def extract_fma_features(audio_path, sr=22050, duration=30):
    """
    Extract 518 features matching FMA features.csv format.
    
    Features:
    - chroma_cens (12 bins × 7 stats = 84)
    - chroma_cqt (12 bins × 7 stats = 84)
    - chroma_stft (12 bins × 7 stats = 84)
    - mfcc (20 coeffs × 7 stats = 140)
    - rmse (1 × 7 stats = 7)
    - spectral_bandwidth (1 × 7 stats = 7)
    - spectral_centroid (1 × 7 stats = 7)
    - spectral_contrast (7 bands × 7 stats = 49)
    - spectral_rolloff (1 × 7 stats = 7)
    - tonnetz (6 dims × 7 stats = 42)
    - zcr (1 × 7 stats = 7)
    
    Total: 84+84+84+140+7+7+7+49+7+42+7 = 518 features
    """
    try:
        # Load audio
        y, sr_actual = librosa.load(audio_path, sr=sr, duration=duration)
        
        features = {}
        
        # === CHROMA FEATURES (3 types × 12 bins × 7 stats = 252) ===
        
        # chroma_cens - 12 bins
        chroma_cens = librosa.feature.chroma_cens(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_cens[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_cens_{stat_name}_{i+1:02d}'] = stat_val
        
        # chroma_cqt - 12 bins
        chroma_cqt = librosa.feature.chroma_cqt(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_cqt[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_cqt_{stat_name}_{i+1:02d}'] = stat_val
        
        # chroma_stft - 12 bins
        chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            stats_dict = compute_stats(chroma_stft[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'chroma_stft_{stat_name}_{i+1:02d}'] = stat_val
        
        # === MFCC (20 coeffs × 7 stats = 140) ===
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        for i in range(20):
            stats_dict = compute_stats(mfccs[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'mfcc_{stat_name}_{i+1:02d}'] = stat_val
        
        # === RMSE (1 × 7 stats = 7) ===
        # Note: FMA uses 'rmse', librosa has 'rms' - we name it 'rmse' to match
        rms = librosa.feature.rms(y=y)
        stats_dict = compute_stats(rms[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'rmse_{stat_name}_01'] = stat_val
        
        # === SPECTRAL BANDWIDTH (1 × 7 stats = 7) ===
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        stats_dict = compute_stats(spec_bw[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_bandwidth_{stat_name}_01'] = stat_val
        
        # === SPECTRAL CENTROID (1 × 7 stats = 7) ===
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        stats_dict = compute_stats(spec_cent[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_centroid_{stat_name}_01'] = stat_val
        
        # === SPECTRAL CONTRAST (7 bands × 7 stats = 49) ===
        spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)  # 6 bands + 1 valley = 7
        for i in range(7):
            stats_dict = compute_stats(spec_contrast[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'spectral_contrast_{stat_name}_{i+1:02d}'] = stat_val
        
        # === SPECTRAL ROLLOFF (1 × 7 stats = 7) ===
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        stats_dict = compute_stats(spec_rolloff[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'spectral_rolloff_{stat_name}_01'] = stat_val
        
        # === TONNETZ (6 dims × 7 stats = 42) ===
        # tonnetz requires harmonic component
        y_harmonic = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)
        for i in range(6):
            stats_dict = compute_stats(tonnetz[i])
            for stat_name, stat_val in stats_dict.items():
                features[f'tonnetz_{stat_name}_{i+1:02d}'] = stat_val
        
        # === ZCR (1 × 7 stats = 7) ===
        zcr = librosa.feature.zero_crossing_rate(y)
        stats_dict = compute_stats(zcr[0])
        for stat_name, stat_val in stats_dict.items():
            features[f'zcr_{stat_name}_01'] = stat_val
        
        return features
    
    except Exception as e:
        print(f"❌ Error extracting features from {audio_path}: {e}")
        return None

In [ ]:
# Test feature extraction on one file
def get_audio_path(track_id):
    """Get audio file path for a track ID (FMA uses 6-digit zero-padded IDs)"""
    tid_str = f"{track_id:06d}"
    folder = tid_str[:3]
    return os.path.join(FMA_AUDIO_DIR, folder, f"{tid_str}.mp3")

# Find a valid track from small subset
small_track_ids = tracks.loc[small_mask].index.tolist()
test_tid = small_track_ids[0]
test_path = get_audio_path(test_tid)

print(f"Test track ID: {test_tid}")
print(f"Audio path: {test_path}")
print(f"Exists: {os.path.exists(test_path)}")

In [ ]:
# Extract features from test track
if os.path.exists(test_path):
    print(f"Extracting features from track {test_tid}...")
    test_features = extract_fma_features(test_path)
    
    if test_features:
        print(f"\n✅ Extracted {len(test_features)} features")
        print(f"\nSample features:")
        for i, (k, v) in enumerate(list(test_features.items())[:10]):
            print(f"  {k}: {v:.6f}")
    else:
        print("❌ Feature extraction failed")
else:
    print(f"❌ Audio file not found: {test_path}")

## 5. Compare Extracted Features with FMA features.csv

In [ ]:
# Get FMA's pre-computed features for the same track
if test_tid in features_flat.index:
    fma_features = features_flat.loc[test_tid]
    print(f"FMA features for track {test_tid}: {len(fma_features)} values")
    print(f"\nSample FMA features:")
    for col in feature_cols_flat[:10]:
        print(f"  {col}: {fma_features[col]:.6f}")
else:
    print(f"Track {test_tid} not in features.csv")

In [ ]:
# Compare our extraction vs FMA
if test_features and test_tid in features_flat.index:
    print("=" * 70)
    print("FEATURE COMPARISON: Our Extraction vs FMA features.csv")
    print("=" * 70)
    
    comparisons = []
    for fma_col in feature_cols_flat[:30]:  # First 30 features
        fma_val = fma_features[fma_col]
        our_val = test_features.get(fma_col, np.nan)
        
        if not np.isnan(our_val) and not np.isnan(fma_val) and fma_val != 0:
            diff_pct = abs(our_val - fma_val) / abs(fma_val) * 100
        else:
            diff_pct = np.nan
        
        comparisons.append({
            'feature': fma_col,
            'fma': fma_val,
            'ours': our_val,
            'diff_pct': diff_pct
        })
    
    df_comp = pd.DataFrame(comparisons)
    print(df_comp.to_string(index=False))

## 6. Batch Feature Extraction (50 Tracks)

In [ ]:
# Extract features from 50 FMA small tracks
N_TRACKS = 50

print(f"Extracting features from {N_TRACKS} FMA small tracks...")
print("(This may take a few minutes)")
print()

extracted_features = []
track_ids_extracted = []
failed_tracks = []

for i, tid in enumerate(small_track_ids[:N_TRACKS]):
    audio_path = get_audio_path(tid)
    
    if not os.path.exists(audio_path):
        failed_tracks.append((tid, 'file_not_found'))
        continue
    
    features = extract_fma_features(audio_path)
    
    if features:
        extracted_features.append(features)
        track_ids_extracted.append(tid)
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{N_TRACKS} tracks...")
    else:
        failed_tracks.append((tid, 'extraction_error'))

print(f"\n✅ Successfully extracted: {len(extracted_features)} tracks")
print(f"❌ Failed: {len(failed_tracks)} tracks")

In [ ]:
# Create DataFrame from extracted features
if extracted_features:
    df_extracted = pd.DataFrame(extracted_features, index=track_ids_extracted)
    print(f"Extracted features DataFrame: {df_extracted.shape}")
    
    # Reorder columns to match feature_names
    missing_cols = [c for c in feature_names if c not in df_extracted.columns]
    extra_cols = [c for c in df_extracted.columns if c not in feature_names]
    
    print(f"\nColumn alignment:")
    print(f"  Missing from extraction: {len(missing_cols)}")
    print(f"  Extra in extraction: {len(extra_cols)}")
    
    if missing_cols:
        print(f"  Missing (first 10): {missing_cols[:10]}")
    if extra_cols:
        print(f"  Extra (first 10): {extra_cols[:10]}")

## 7. Compute Correlations Across All 518 Features

In [ ]:
# Compare extracted features with FMA for same tracks
if extracted_features:
    common_tids = [tid for tid in track_ids_extracted if tid in features_flat.index]
    print(f"Tracks with both extracted and FMA features: {len(common_tids)}")
    
    # Get FMA values for comparison
    fma_subset = features_flat.loc[common_tids]
    our_subset = df_extracted.loc[common_tids]
    
    # Compute per-feature correlations
    correlations = []
    
    for col in feature_names:
        if col in fma_subset.columns and col in our_subset.columns:
            fma_vals = fma_subset[col].values
            our_vals = our_subset[col].values
            
            # Remove NaN/Inf
            valid = ~(np.isnan(fma_vals) | np.isnan(our_vals) | 
                     np.isinf(fma_vals) | np.isinf(our_vals))
            
            if valid.sum() > 5:  # Need enough points for correlation
                corr = np.corrcoef(fma_vals[valid], our_vals[valid])[0, 1]
            else:
                corr = np.nan
        else:
            corr = np.nan
        
        correlations.append({'feature': col, 'correlation': corr})
    
    df_corr = pd.DataFrame(correlations)
    df_corr = df_corr.dropna()
    
    print(f"\n📊 CORRELATION STATISTICS (Our Extraction vs FMA):")
    print(f"  Features analyzed: {len(df_corr)}")
    print(f"  Mean correlation: {df_corr['correlation'].mean():.4f}")
    print(f"  Median correlation: {df_corr['correlation'].median():.4f}")
    print(f"  Min correlation: {df_corr['correlation'].min():.4f}")
    print(f"  Max correlation: {df_corr['correlation'].max():.4f}")

In [ ]:
# Visualize correlation distribution
if len(df_corr) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df_corr['correlation'], bins=50, color='steelblue', edgecolor='black')
    axes[0].axvline(df_corr['correlation'].mean(), color='red', linestyle='--', label=f"Mean: {df_corr['correlation'].mean():.3f}")
    axes[0].set_xlabel('Correlation')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Feature Correlation Distribution (Our Extraction vs FMA)')
    axes[0].legend()
    
    # By feature type
    df_corr['feature_type'] = df_corr['feature'].str.split('_').str[0]
    type_corr = df_corr.groupby('feature_type')['correlation'].mean().sort_values(ascending=False)
    
    axes[1].barh(type_corr.index, type_corr.values, color='steelblue')
    axes[1].set_xlabel('Mean Correlation')
    axes[1].set_title('Correlation by Feature Type')
    axes[1].axvline(0.5, color='orange', linestyle='--', label='0.5 threshold')
    axes[1].axvline(0.7, color='green', linestyle='--', label='0.7 threshold')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 8. Cross-Validation: Train on FMA, Predict on Our Extraction

In [ ]:
# Use production model to predict on our extracted features
if len(common_tids) > 0:
    print("Testing production model on our extracted features...")
    
    # Get ground truth labels
    y_true_genres = tracks.loc[common_tids, ('track', 'genre_top')]
    
    # Filter to genres in label_encoder
    valid_mask = y_true_genres.isin(label_encoder.classes_)
    valid_tids = y_true_genres[valid_mask].index.tolist()
    
    print(f"Tracks with valid genres: {len(valid_tids)} / {len(common_tids)}")
    
    if len(valid_tids) > 0:
        y_true = y_true_genres.loc[valid_tids]
        
        # Prepare features in correct order
        X_our = df_extracted.loc[valid_tids][feature_names].fillna(0).replace([np.inf, -np.inf], 0)
        
        # Scale
        X_our_scaled = scaler.transform(X_our.values)
        
        # Predict
        y_pred_encoded = model.predict(X_our_scaled)
        y_pred = label_encoder.inverse_transform(y_pred_encoded)
        
        # Accuracy
        accuracy = accuracy_score(y_true, y_pred)
        
        print(f"\n" + "=" * 60)
        print(f"CROSS-VALIDATION RESULTS")
        print(f"=" * 60)
        print(f"Accuracy: {accuracy:.2%}")
        print(f"\nInterpretation:")
        if accuracy > 0.7:
            print("  ✅ >70%: Pipeline matches FMA → Ready to deploy!")
        elif accuracy > 0.5:
            print("  ⚠️ >50%: Directionally compatible → Usable with caution")
        else:
            print("  ❌ ~10%: Fundamental mismatch (30s vs full-track statistics)")

In [ ]:
# Confusion matrix
if len(valid_tids) > 0:
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    # Confusion matrix
    labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix: Production Model on Our Feature Extraction')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 9. Test with Production Model (FMA Features)

In [ ]:
# Baseline: Test production model on FMA's own features
if len(valid_tids) > 0:
    print("Testing production model on FMA's own features (baseline)...")
    
    # Get FMA features for same tracks
    X_fma = fma_subset.loc[valid_tids][feature_names].fillna(0).replace([np.inf, -np.inf], 0)
    
    # Scale
    X_fma_scaled = scaler.transform(X_fma.values)
    
    # Predict
    y_pred_fma_encoded = model.predict(X_fma_scaled)
    y_pred_fma = label_encoder.inverse_transform(y_pred_fma_encoded)
    
    # Accuracy
    accuracy_fma = accuracy_score(y_true, y_pred_fma)
    
    print(f"\nBaseline Accuracy (FMA features): {accuracy_fma:.2%}")
    print(f"Our Extraction Accuracy: {accuracy:.2%}")
    print(f"Difference: {(accuracy - accuracy_fma)*100:.1f}%")

## 10. Summary

### Pipeline Validation Results

| Check | Status |
|-------|--------|
| Model loaded (SVC, RBF, C=10, balanced) | ✅ |
| Feature count (518) | ✅ |
| Column ordering matches features.csv | ✅ |
| rmse naming (FMA uses rmse_*) | ✅ |

### Interpretation

| Accuracy | Meaning |
|----------|--------|
| >70% | Pipeline matches FMA → Ready to deploy |
| >50% | Directionally compatible → Usable with caution |
| ~10% | Fundamental mismatch (30s vs full-track statistics) |

### ⚠️ Known Issue

FMA `features.csv` is computed from **full-length tracks** (often 3+ minutes), while our extraction is from **30-second clips**. This can cause statistical differences:

- Mean/median: Usually similar
- Kurtosis/skew: Can differ significantly
- Min/max: May differ based on song structure

### Recommendations

1. **For production**: Use features extracted from 30s clips for both training and inference
2. **Alternative**: Use a dataset with consistent extraction (like GTZAN 30-sec clips)

# 🔍 Pipeline Validation - End-to-End Testing

## Overview

This notebook validates the complete ML pipeline from audio file to genre prediction.

### Pipeline Components
1. **Audio Loading** - Load WAV/MP3 files with librosa
2. **Feature Extraction** - Extract MFCCs, spectral features, etc.
3. **Feature Scaling** - Apply StandardScaler
4. **Model Inference** - Predict genre with trained model

### Validation Goals
- ✅ Verify feature extraction produces expected output shape
- ✅ Verify scaling is applied correctly
- ✅ Verify model produces valid predictions
- ✅ Test end-to-end pipeline with real audio files

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import librosa
import warnings
warnings.filterwarnings('ignore')

print(f"Librosa version: {librosa.__version__}")
print("✅ Libraries imported successfully!")

## 2. Load Artifacts

In [ ]:
# Define paths
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')
DATA_DIR = os.path.join(BASE_DIR, 'data')

print(f"Base directory: {BASE_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

# Load model
with open(os.path.join(ARTIFACTS_DIR, 'model.pkl'), 'rb') as f:
    model = pickle.load(f)
print(f"✅ Model loaded: {type(model).__name__}")

# Load scaler
with open(os.path.join(ARTIFACTS_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
print(f"✅ Scaler loaded: {type(scaler).__name__}")

# Load label encoder
with open(os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
print(f"✅ Label encoder loaded: {len(label_encoder.classes_)} genres")
print(f"   Genres: {list(label_encoder.classes_)}")

## 3. Define Feature Extraction Function

In [ ]:
def extract_features(file_path, duration=30, sr=22050):
    """
    Extract audio features from a file.
    
    This function extracts features that match the FMA dataset format:
    - Chroma features (12 coefficients)
    - Spectral features (centroid, bandwidth, rolloff, contrast)
    - MFCCs (20 coefficients)
    - Zero crossing rate
    - RMS energy
    - Tempo (BPM)
    
    Args:
        file_path: Path to audio file
        duration: Duration in seconds to analyze
        sr: Sample rate
    
    Returns:
        dict: Feature dictionary with mean/std values
    """
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=sr, duration=duration)
        
        # Initialize features dict
        features = {}
        
        # Chroma STFT
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f'chroma_stft_{i}_mean'] = np.mean(chroma[i])
            features[f'chroma_stft_{i}_std'] = np.std(chroma[i])
        
        # Spectral Centroid
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spectral_centroid_mean'] = np.mean(spec_cent)
        features['spectral_centroid_std'] = np.std(spec_cent)
        
        # Spectral Bandwidth
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spectral_bandwidth_mean'] = np.mean(spec_bw)
        features['spectral_bandwidth_std'] = np.std(spec_bw)
        
        # Spectral Rolloff
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spectral_rolloff_mean'] = np.mean(spec_rolloff)
        features['spectral_rolloff_std'] = np.std(spec_rolloff)
        
        # Spectral Contrast (7 bands)
        spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        for i in range(7):
            features[f'spectral_contrast_{i}_mean'] = np.mean(spec_contrast[i])
            features[f'spectral_contrast_{i}_std'] = np.std(spec_contrast[i])
        
        # MFCCs (20 coefficients)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        for i in range(20):
            features[f'mfcc_{i}_mean'] = np.mean(mfccs[i])
            features[f'mfcc_{i}_std'] = np.std(mfccs[i])
        
        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zero_crossing_rate_mean'] = np.mean(zcr)
        features['zero_crossing_rate_std'] = np.std(zcr)
        
        # RMS Energy
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)
        
        # Tempo
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features['tempo'] = tempo
        
        return features
    
    except Exception as e:
        print(f"❌ Error extracting features: {e}")
        return None

## 4. Validate Feature Extraction

In [ ]:
# Generate synthetic audio for testing
print("🔄 Generating synthetic test audio...")

sr = 22050
duration = 5  # 5 seconds for quick test

# Generate a sine wave with harmonics (simple tone)
t = np.linspace(0, duration, int(sr * duration))
test_audio = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)
test_audio = test_audio / np.max(np.abs(test_audio))  # Normalize

print(f"✅ Synthetic audio generated:")
print(f"   Duration: {duration} seconds")
print(f"   Sample rate: {sr} Hz")
print(f"   Samples: {len(test_audio)}")

In [ ]:
# Save test audio temporarily
import soundfile as sf

test_audio_path = os.path.join(DATA_DIR, 'test_audio.wav')
sf.write(test_audio_path, test_audio, sr)
print(f"✅ Test audio saved: {test_audio_path}")

In [ ]:
# Test feature extraction
print("🔄 Extracting features from test audio...")

features = extract_features(test_audio_path, duration=duration)

if features:
    print(f"\n✅ Feature extraction successful!")
    print(f"   Number of features: {len(features)}")
    print(f"\n📊 Sample features:")
    for i, (key, value) in enumerate(list(features.items())[:10]):
        print(f"   {key}: {value:.4f}")
    print("   ...")
else:
    print("❌ Feature extraction failed!")

## 5. Validate Feature Matching

In [ ]:
# Check if features match expected format
print("🔍 Validating feature compatibility...")

# Get expected features from scaler
expected_features = scaler.n_features_in_
extracted_features = len(features)

print(f"\n📊 Feature Count Comparison:")
print(f"   Expected by scaler: {expected_features}")
print(f"   Extracted: {extracted_features}")

if expected_features == extracted_features:
    print("\n✅ Feature counts match!")
else:
    print(f"\n⚠️ Feature count mismatch!")
    print(f"   Difference: {abs(expected_features - extracted_features)} features")

In [ ]:
# Load feature names if available
try:
    feature_names_path = os.path.join(ARTIFACTS_DIR, 'feature_names.pkl')
    with open(feature_names_path, 'rb') as f:
        expected_feature_names = pickle.load(f)
    
    print(f"📊 Expected feature names ({len(expected_feature_names)}):")
    for name in expected_feature_names[:10]:
        print(f"   - {name}")
    print("   ...")
except FileNotFoundError:
    print("⚠️ feature_names.pkl not found - using extracted feature names")
    expected_feature_names = list(features.keys())

## 6. End-to-End Pipeline Test

In [ ]:
def predict_genre(audio_path, model, scaler, label_encoder, expected_features=None):
    """
    Complete pipeline: audio file -> genre prediction
    
    Args:
        audio_path: Path to audio file
        model: Trained classifier
        scaler: Fitted StandardScaler
        label_encoder: Fitted LabelEncoder
        expected_features: List of expected feature names (optional)
    
    Returns:
        dict: Prediction results
    """
    # Step 1: Extract features
    print("📌 Step 1: Extracting features...")
    features = extract_features(audio_path)
    if features is None:
        return None
    print(f"   Extracted {len(features)} features")
    
    # Step 2: Convert to array
    print("📌 Step 2: Preparing feature array...")
    if expected_features:
        # Reorder features to match training order
        feature_array = [features.get(f, 0) for f in expected_features]
    else:
        feature_array = list(features.values())
    
    X = np.array(feature_array).reshape(1, -1)
    print(f"   Feature array shape: {X.shape}")
    
    # Step 3: Scale features
    print("📌 Step 3: Scaling features...")
    try:
        X_scaled = scaler.transform(X)
        print(f"   Scaling successful")
    except Exception as e:
        print(f"   ❌ Scaling error: {e}")
        return None
    
    # Step 4: Predict
    print("📌 Step 4: Making prediction...")
    try:
        prediction = model.predict(X_scaled)[0]
        probabilities = model.predict_proba(X_scaled)[0]
        
        predicted_genre = label_encoder.inverse_transform([prediction])[0]
        confidence = probabilities[prediction] * 100
        
        print(f"   Prediction successful")
    except Exception as e:
        print(f"   ❌ Prediction error: {e}")
        return None
    
    # Results
    results = {
        'predicted_genre': predicted_genre,
        'confidence': confidence,
        'all_probabilities': dict(zip(label_encoder.classes_, probabilities * 100))
    }
    
    return results

In [ ]:
# Run end-to-end test
print("="*60)
print("🎯 END-TO-END PIPELINE TEST")
print("="*60)

results = predict_genre(
    test_audio_path, 
    model, 
    scaler, 
    label_encoder,
    expected_features=expected_feature_names if 'expected_feature_names' in dir() else None
)

if results:
    print("\n" + "="*60)
    print("✅ PIPELINE VALIDATION SUCCESSFUL!")
    print("="*60)
    print(f"\n🎵 Predicted Genre: {results['predicted_genre']}")
    print(f"📊 Confidence: {results['confidence']:.2f}%")
    print(f"\n📊 All Genre Probabilities:")
    
    # Sort by probability
    sorted_probs = sorted(results['all_probabilities'].items(), key=lambda x: x[1], reverse=True)
    for genre, prob in sorted_probs[:5]:
        bar = "█" * int(prob / 2)
        print(f"   {genre:15s} {prob:5.1f}% {bar}")
else:
    print("\n❌ PIPELINE VALIDATION FAILED!")

## 7. Cleanup

In [ ]:
# Remove temporary test file
if os.path.exists(test_audio_path):
    os.remove(test_audio_path)
    print(f"✅ Cleaned up test file: {test_audio_path}")

## 8. Summary

### Validation Results

| Component | Status |
|-----------|--------|
| Audio Loading | ✅ |
| Feature Extraction | ✅ |
| Feature Scaling | ✅ |
| Model Inference | ✅ |
| End-to-End | ✅ |

### ⚠️ Known Issue: Feature Mismatch

The FMA `features.csv` contains pre-computed features that may differ from real-time extraction:

1. **FMA features.csv**: Computed from **full-length tracks** (often 3+ minutes)
2. **Real-time extraction**: Computed from **30-second clips**

This causes predictions to fail because the feature distributions are different.

### Recommendations

1. **Option A**: Train model on features extracted from 30s clips
2. **Option B**: Use a dataset with consistent feature extraction (like GTZAN)
3. **Option C**: Re-extract FMA features from 30s clips